In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

cases = pd.read_csv(DATA_DIR / "cases.csv")
points = pd.read_csv(DATA_DIR / "points.csv")
topics = pd.read_csv(DATA_DIR / "topics.csv")
services = pd.read_csv(DATA_DIR / "services.csv")

In [32]:
approved_points = points[points["status"] == "approved"].copy()

print("Total approved points:", len(approved_points))

Total approved points: 23968


In [45]:
approved_points = approved_points[
    
    approved_points["quote_text"].notna()
].copy()

print("Points with valid quote_text:", len(approved_points))

Points with valid quote_text: 22498


In [46]:
dataset = approved_points.merge(
    cases[["id", "classification", "topic_id"]],
    left_on="case_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

In [47]:
dataset

,id_x,rank,title,source,status,analysis,created_at,updated_at,service_id,quote_text,...,old_id,point_change,quote_start,quote_end,service_needs_rating_update,document_id,annotation_ref,id_y,classification,topic_id
0,6132,0,The court of law governing the terms is in a j...,https://stackoverflow.com/legal/terms-of-servi...,approved,Generated through the annotate view,2019-03-19 12:45:34.493941,2023-07-11 06:33:59.625461,312,Both you and Stack Overflow hereby irrevocably...,...,NaN,NaN,20471.0,20749.0,False,352.0,6zrawh-0Ee66pFffOexWXQ,242,bad,44
1,9860,0,You can request access and deletion of persona...,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:59.854515,2023-07-11 06:33:59.728921,1700,Sie haben jederzeit das Recht unentgeltlich Au...,...,NaN,NaN,5669.0,5911.0,False,1625.0,60qawB-0Ee66pG97_ve0ew,195,good,34
2,9859,0,Your personal data is used for limited purposes,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:25.245754,2023-07-11 06:33:59.813688,1700,"Ein Teil der Daten wird erhoben, um eine fehle...",...,NaN,NaN,5428.0,5602.0,False,1625.0,61eQ6h-0Ee66pIu8RRm9-w,196,good,41
3,4957,0,The service is not responsible for linked or (...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:18:49.671591,2023-07-11 06:33:59.898468,820,Please note that we are not responsible for ot...,...,NaN,NaN,9839.0,9914.0,False,725.0,62SBJB-0Ee66pC-R84wfHQ,278,neutral,32
4,4960,0,The service provides details about what kinds ...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:29:27.41984,2023-07-11 06:33:59.984025,820,"Your name, email address and optional informat...",...,NaN,NaN,8241.0,8326.0,False,725.0,63GOoB-0Ee66pItWc4iOIw,228,good,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22493,8825,0,This service is only available to users of 16 ...,https://letterboxd.com/legal/terms-of-use/,approved,Generated through the annotate view,2020-01-02 20:32:15.303053,2023-07-11 06:33:59.075793,2074,In order to use the services provided on the S...,...,NaN,NaN,1126.0,1217.0,False,2262.0,6ubeLB-0Ee66pGdXeniyNw,152,neutral,31
22494,12892,0,Your personal data is aggregated into statistics,https://www.w3schools.com/about/about_privacy.asp,approved,Generated through the annotate view,2020-12-18 05:51:54.250795,2023-07-11 06:33:59.16018,319,<p>We use the Device Information that we colle...,...,NaN,NaN,2526.0,2803.0,False,227.0,6vPZxB-0Ee66pJvuGO2N-A,190,neutral,41
22495,4959,0,The terms for this service are easy to read,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:25:00.350444,2023-07-11 06:33:59.245709,820,"Privacy, Security, Cookies and Warranty",...,NaN,NaN,7865.0,7906.0,False,725.0,6wDbnB-0Ee66pBvuLWGsGA,199,good,36
22496,5031,0,The service will resist legal requests for use...,https://www.bitsoffreedom.nl/privacy-en-voorwa...,approved,Generated through the annotate view,2018-11-30 08:26:15.966235,2023-07-11 06:33:59.349994,844,We zullen je persoonsgegevens nooit aan derden...,...,NaN,NaN,13347.0,13773.0,False,757.0,6xCwMB-0Ee66pHeL3_Ufng,305,good,28


In [48]:
dataset.columns

Index(['id_x', 'rank', 'title', 'source', 'status', 'analysis', 'created_at',
       'updated_at', 'service_id', 'quote_text', 'case_id', 'old_id',
       'point_change', 'quote_start', 'quote_end',
       'service_needs_rating_update', 'document_id', 'annotation_ref', 'id_y',
       'classification', 'topic_id'],
      dtype='str')

In [49]:
dataset = dataset.rename(columns={
    "id_x": "point_id"
})

dataset = dataset.drop(columns=["id_y"])

print(dataset.columns.tolist())

['point_id', 'rank', 'title', 'source', 'status', 'analysis', 'created_at', 'updated_at', 'service_id', 'quote_text', 'case_id', 'old_id', 'point_change', 'quote_start', 'quote_end', 'service_needs_rating_update', 'document_id', 'annotation_ref', 'classification', 'topic_id']


In [50]:
dataset = dataset.merge(
    topics[["id", "title"]],
    left_on="topic_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

In [51]:
dataset

,point_id,rank,title_x,source,status,analysis,created_at,updated_at,service_id,quote_text,...,point_change,quote_start,quote_end,service_needs_rating_update,document_id,annotation_ref,classification,topic_id,id,title_y
0,6132,0,The court of law governing the terms is in a j...,https://stackoverflow.com/legal/terms-of-servi...,approved,Generated through the annotate view,2019-03-19 12:45:34.493941,2023-07-11 06:33:59.625461,312,Both you and Stack Overflow hereby irrevocably...,...,NaN,20471.0,20749.0,False,352.0,6zrawh-0Ee66pFffOexWXQ,bad,44,44,Jurisdiction and governing laws
1,9860,0,You can request access and deletion of persona...,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:59.854515,2023-07-11 06:33:59.728921,1700,Sie haben jederzeit das Recht unentgeltlich Au...,...,NaN,5669.0,5911.0,False,1625.0,60qawB-0Ee66pG97_ve0ew,good,34,34,User Choice
2,9859,0,Your personal data is used for limited purposes,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:25.245754,2023-07-11 06:33:59.813688,1700,"Ein Teil der Daten wird erhoben, um eine fehle...",...,NaN,5428.0,5602.0,False,1625.0,61eQ6h-0Ee66pIu8RRm9-w,good,41,41,Personal Data
3,4957,0,The service is not responsible for linked or (...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:18:49.671591,2023-07-11 06:33:59.898468,820,Please note that we are not responsible for ot...,...,NaN,9839.0,9914.0,False,725.0,62SBJB-0Ee66pC-R84wfHQ,neutral,32,32,Guarantee
4,4960,0,The service provides details about what kinds ...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:29:27.41984,2023-07-11 06:33:59.984025,820,"Your name, email address and optional informat...",...,NaN,8241.0,8326.0,False,725.0,63GOoB-0Ee66pItWc4iOIw,good,36,36,Transparency
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22493,8825,0,This service is only available to users of 16 ...,https://letterboxd.com/legal/terms-of-use/,approved,Generated through the annotate view,2020-01-02 20:32:15.303053,2023-07-11 06:33:59.075793,2074,In order to use the services provided on the S...,...,NaN,1126.0,1217.0,False,2262.0,6ubeLB-0Ee66pGdXeniyNw,neutral,31,31,Governance
22494,12892,0,Your personal data is aggregated into statistics,https://www.w3schools.com/about/about_privacy.asp,approved,Generated through the annotate view,2020-12-18 05:51:54.250795,2023-07-11 06:33:59.16018,319,<p>We use the Device Information that we colle...,...,NaN,2526.0,2803.0,False,227.0,6vPZxB-0Ee66pJvuGO2N-A,neutral,41,41,Personal Data
22495,4959,0,The terms for this service are easy to read,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:25:00.350444,2023-07-11 06:33:59.245709,820,"Privacy, Security, Cookies and Warranty",...,NaN,7865.0,7906.0,False,725.0,6wDbnB-0Ee66pBvuLWGsGA,good,36,36,Transparency
22496,5031,0,The service will resist legal requests for use...,https://www.bitsoffreedom.nl/privacy-en-voorwa...,approved,Generated through the annotate view,2018-11-30 08:26:15.966235,2023-07-11 06:33:59.349994,844,We zullen je persoonsgegevens nooit aan derden...,...,NaN,13347.0,13773.0,False,757.0,6xCwMB-0Ee66pHeL3_Ufng,good,28,28,Law and Government Requests


In [55]:
dataset = dataset.rename(columns={
    "title_y": "topic_title"
})

dataset = dataset.drop(columns=["id"])

KeyError: "['id'] not found in axis"

In [56]:
dataset

,point_id,rank,title_x,source,status,analysis,created_at,updated_at,service_id,quote_text,...,old_id,point_change,quote_start,quote_end,service_needs_rating_update,document_id,annotation_ref,classification,topic_id,topic_title
0,6132,0,The court of law governing the terms is in a j...,https://stackoverflow.com/legal/terms-of-servi...,approved,Generated through the annotate view,2019-03-19 12:45:34.493941,2023-07-11 06:33:59.625461,312,Both you and Stack Overflow hereby irrevocably...,...,NaN,NaN,20471.0,20749.0,False,352.0,6zrawh-0Ee66pFffOexWXQ,bad,44,Jurisdiction and governing laws
1,9860,0,You can request access and deletion of persona...,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:59.854515,2023-07-11 06:33:59.728921,1700,Sie haben jederzeit das Recht unentgeltlich Au...,...,NaN,NaN,5669.0,5911.0,False,1625.0,60qawB-0Ee66pG97_ve0ew,good,34,User Choice
2,9859,0,Your personal data is used for limited purposes,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:25.245754,2023-07-11 06:33:59.813688,1700,"Ein Teil der Daten wird erhoben, um eine fehle...",...,NaN,NaN,5428.0,5602.0,False,1625.0,61eQ6h-0Ee66pIu8RRm9-w,good,41,Personal Data
3,4957,0,The service is not responsible for linked or (...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:18:49.671591,2023-07-11 06:33:59.898468,820,Please note that we are not responsible for ot...,...,NaN,NaN,9839.0,9914.0,False,725.0,62SBJB-0Ee66pC-R84wfHQ,neutral,32,Guarantee
4,4960,0,The service provides details about what kinds ...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:29:27.41984,2023-07-11 06:33:59.984025,820,"Your name, email address and optional informat...",...,NaN,NaN,8241.0,8326.0,False,725.0,63GOoB-0Ee66pItWc4iOIw,good,36,Transparency
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22493,8825,0,This service is only available to users of 16 ...,https://letterboxd.com/legal/terms-of-use/,approved,Generated through the annotate view,2020-01-02 20:32:15.303053,2023-07-11 06:33:59.075793,2074,In order to use the services provided on the S...,...,NaN,NaN,1126.0,1217.0,False,2262.0,6ubeLB-0Ee66pGdXeniyNw,neutral,31,Governance
22494,12892,0,Your personal data is aggregated into statistics,https://www.w3schools.com/about/about_privacy.asp,approved,Generated through the annotate view,2020-12-18 05:51:54.250795,2023-07-11 06:33:59.16018,319,<p>We use the Device Information that we colle...,...,NaN,NaN,2526.0,2803.0,False,227.0,6vPZxB-0Ee66pJvuGO2N-A,neutral,41,Personal Data
22495,4959,0,The terms for this service are easy to read,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:25:00.350444,2023-07-11 06:33:59.245709,820,"Privacy, Security, Cookies and Warranty",...,NaN,NaN,7865.0,7906.0,False,725.0,6wDbnB-0Ee66pBvuLWGsGA,good,36,Transparency
22496,5031,0,The service will resist legal requests for use...,https://www.bitsoffreedom.nl/privacy-en-voorwa...,approved,Generated through the annotate view,2018-11-30 08:26:15.966235,2023-07-11 06:33:59.349994,844,We zullen je persoonsgegevens nooit aan derden...,...,NaN,NaN,13347.0,13773.0,False,757.0,6xCwMB-0Ee66pHeL3_Ufng,good,28,Law and Government Requests


In [57]:
print("Missing classification:", dataset["classification"].isna().sum())
print("Missing topic_id:", dataset["topic_id"].isna().sum())
print("Missing topic_title:", dataset["topic_title"].isna().sum())

Missing classification: 0
Missing topic_id: 0
Missing topic_title: 0


In [58]:
label_map = {
    "good": "Safe",
    "neutral": "Neutral",
    "bad": "Risky",
    "blocker": "Risky"
}

dataset["label"] = dataset["classification"].map(label_map)

In [59]:
dataset

,point_id,rank,title_x,source,status,analysis,created_at,updated_at,service_id,quote_text,...,point_change,quote_start,quote_end,service_needs_rating_update,document_id,annotation_ref,classification,topic_id,topic_title,label
0,6132,0,The court of law governing the terms is in a j...,https://stackoverflow.com/legal/terms-of-servi...,approved,Generated through the annotate view,2019-03-19 12:45:34.493941,2023-07-11 06:33:59.625461,312,Both you and Stack Overflow hereby irrevocably...,...,NaN,20471.0,20749.0,False,352.0,6zrawh-0Ee66pFffOexWXQ,bad,44,Jurisdiction and governing laws,Risky
1,9860,0,You can request access and deletion of persona...,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:59.854515,2023-07-11 06:33:59.728921,1700,Sie haben jederzeit das Recht unentgeltlich Au...,...,NaN,5669.0,5911.0,False,1625.0,60qawB-0Ee66pG97_ve0ew,good,34,User Choice,Safe
2,9859,0,Your personal data is used for limited purposes,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:25.245754,2023-07-11 06:33:59.813688,1700,"Ein Teil der Daten wird erhoben, um eine fehle...",...,NaN,5428.0,5602.0,False,1625.0,61eQ6h-0Ee66pIu8RRm9-w,good,41,Personal Data,Safe
3,4957,0,The service is not responsible for linked or (...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:18:49.671591,2023-07-11 06:33:59.898468,820,Please note that we are not responsible for ot...,...,NaN,9839.0,9914.0,False,725.0,62SBJB-0Ee66pC-R84wfHQ,neutral,32,Guarantee,Neutral
4,4960,0,The service provides details about what kinds ...,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:29:27.41984,2023-07-11 06:33:59.984025,820,"Your name, email address and optional informat...",...,NaN,8241.0,8326.0,False,725.0,63GOoB-0Ee66pItWc4iOIw,good,36,Transparency,Safe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22493,8825,0,This service is only available to users of 16 ...,https://letterboxd.com/legal/terms-of-use/,approved,Generated through the annotate view,2020-01-02 20:32:15.303053,2023-07-11 06:33:59.075793,2074,In order to use the services provided on the S...,...,NaN,1126.0,1217.0,False,2262.0,6ubeLB-0Ee66pGdXeniyNw,neutral,31,Governance,Neutral
22494,12892,0,Your personal data is aggregated into statistics,https://www.w3schools.com/about/about_privacy.asp,approved,Generated through the annotate view,2020-12-18 05:51:54.250795,2023-07-11 06:33:59.16018,319,<p>We use the Device Information that we colle...,...,NaN,2526.0,2803.0,False,227.0,6vPZxB-0Ee66pJvuGO2N-A,neutral,41,Personal Data,Neutral
22495,4959,0,The terms for this service are easy to read,https://www.webseodesigners.com/,approved,Generated through the annotate view,2018-11-10 11:25:00.350444,2023-07-11 06:33:59.245709,820,"Privacy, Security, Cookies and Warranty",...,NaN,7865.0,7906.0,False,725.0,6wDbnB-0Ee66pBvuLWGsGA,good,36,Transparency,Safe
22496,5031,0,The service will resist legal requests for use...,https://www.bitsoffreedom.nl/privacy-en-voorwa...,approved,Generated through the annotate view,2018-11-30 08:26:15.966235,2023-07-11 06:33:59.349994,844,We zullen je persoonsgegevens nooit aan derden...,...,NaN,13347.0,13773.0,False,757.0,6xCwMB-0Ee66pHeL3_Ufng,good,28,Law and Government Requests,Safe


In [60]:
print(dataset["classification"].value_counts())
print()
print(dataset["label"].value_counts())

classification
neutral    9759
bad        6558
good       5724
blocker     457
Name: count, dtype: int64

label
Neutral    9759
Risky      7015
Safe       5724
Name: count, dtype: int64


In [61]:
print("Missing labels:", dataset["label"].isna().sum())

Missing labels: 0


In [62]:
print(
    "Duplicate point IDs:",
    dataset["point_id"].duplicated().sum()
)

Duplicate point IDs: 0


In [63]:
print(
    "Duplicate quote texts:",
    dataset["quote_text"].duplicated().sum()
)

Duplicate quote texts: 344


In [64]:
dataset = dataset[dataset["quote_text"].str.strip() != ""]

In [65]:
final_dataset = dataset[
    [
        "point_id",
        "quote_text",
        "case_id",
        "classification",
        "label",
        "topic_id",
        "topic_title",
        "service_id",
        "document_id"
    ]
].copy()

In [66]:
final_dataset

,point_id,quote_text,case_id,classification,label,topic_id,topic_title,service_id,document_id
0,6132,Both you and Stack Overflow hereby irrevocably...,242,bad,Risky,44,Jurisdiction and governing laws,312,352.0
1,9860,Sie haben jederzeit das Recht unentgeltlich Au...,195,good,Safe,34,User Choice,1700,1625.0
2,9859,"Ein Teil der Daten wird erhoben, um eine fehle...",196,good,Safe,41,Personal Data,1700,1625.0
3,4957,Please note that we are not responsible for ot...,278,neutral,Neutral,32,Guarantee,820,725.0
4,4960,"Your name, email address and optional informat...",228,good,Safe,36,Transparency,820,725.0
...,...,...,...,...,...,...,...,...,...
22493,8825,In order to use the services provided on the S...,152,neutral,Neutral,31,Governance,2074,2262.0
22494,12892,<p>We use the Device Information that we colle...,190,neutral,Neutral,41,Personal Data,319,227.0
22495,4959,"Privacy, Security, Cookies and Warranty",199,good,Safe,36,Transparency,820,725.0
22496,5031,We zullen je persoonsgegevens nooit aan derden...,305,good,Safe,28,Law and Government Requests,844,757.0


In [67]:
final_dataset["quote_text"].isna().sum()

np.int64(0)

In [68]:
final_dataset[final_dataset["quote_text"].str.strip() == ""]

,point_id,quote_text,case_id,classification,label,topic_id,topic_title,service_id,document_id


In [69]:
final_dataset["label"].value_counts()

label
Neutral    9759
Risky      7015
Safe       5723
Name: count, dtype: int64

In [70]:
dataset.isna().sum()

point_id                           0
rank                               0
title_x                            0
source                           115
status                             0
analysis                          81
created_at                         0
updated_at                         0
service_id                         0
quote_text                         0
case_id                            0
old_id                         22351
point_change                   22214
quote_start                      237
quote_end                        237
service_needs_rating_update        0
document_id                       11
annotation_ref                    87
classification                     0
topic_id                           0
topic_title                        0
label                              0
dtype: int64

In [71]:
final_dataset = final_dataset.reset_index(drop=True)

In [73]:
final_dataset.to_csv("../data/processed/clauses_clean.csv", index=False)